In [19]:
import csv
import pandas as pd

class PipelineMl:
    def __init__(self, data=None):
        self._data = data
        self._feedbacks = None
        self.df_feedbacks = None  # ← atributo público

    def dados(self):
        data = [
            ["Id", "Nome", "Data_Nascimento", "Número", "Email", "Renda"],
            [1, "Ana",     "1995-06-15", "123456789", "ana@email.com",     "R$5000"],
            [2, "Bruno",   "1989-03-22", "987654321", "bruno@email.com",   "R$6000"],
            [3, "Melissa", "2001-12-08", "555555555", "melissa@email.com", "R$7000"],
            [4, "Carlos",  "1978-09-30", "111111111", "carlos@email.com",  "R$8000"],
            [5, "Fernanda","1992-07-14", "222222222", "fernanda@email.com","R$9000"],
            [6, "Caio",    None,         "333333333", "caio@email.com",    "R$10000"],
        ]
        feedbacks = [
            ["Id", "Feedback"],
            [1, "Ótimo serviço!"],
            [2, "Poderia ser melhor."],
            [3, "Excelente atendimento!"],
            [4, "Não estou satisfeito."],
            [5, "Muito bom!"],
            [6, "Nenhum feedback."],
        ]
        self._data      = data
        self._feedbacks = feedbacks
        return data, feedbacks

    def _salvar_csvs(self):
        with open("dados.csv", mode="w", encoding="utf-8", newline="") as f:
            csv.writer(f).writerows(self._data)
        with open("feedbacks.csv", mode="w", encoding="utf-8", newline="") as f:
            csv.writer(f).writerows(self._feedbacks)

    def processar_dados(self):
        if self._data is None:
            self.dados()
        self._salvar_csvs()
        print("Pipeline de Machine Learning concluída com sucesso!\n")

        with open("dados.csv", mode="r", encoding="utf-8") as arq1, \
             open("feedbacks.csv", mode="r", encoding="utf-8") as arq2:
            for linha, linha2 in zip(csv.reader(arq1), csv.reader(arq2)):
                print("Dados:", linha)
                print("Feedback: ", linha2)

        self.df_feedbacks = pd.read_csv("feedbacks.csv")  # ← salva no atributo
        print("\nDataFrame de Feedbacks:")
        print(self.df_feedbacks)


Limpeza os Dados


In [20]:
dt = PipelineMl()
dt.dados()
dt.processar_dados()

# Limpeza dos dados principais
dt_raw = pd.read_csv('dados.csv')
linhas_filtradas = dt_raw[dt_raw['Data_Nascimento'] != 'None'].copy()
linhas_filtradas['Data_Nascimento'] = (
    linhas_filtradas['Data_Nascimento']
    .replace('None', 'Não Informado')
    .fillna('Não Informado')
)

dt_limpo = linhas_filtradas[linhas_filtradas['Renda'].str.contains("R$", regex=False)].copy()
dt_limpo['Renda'] = (
    dt_limpo['Renda']
    .str.replace("R$", "", regex=False)
    .str.strip()
    .astype(float)
)

Pipeline de Machine Learning concluída com sucesso!

Dados: ['Id', 'Nome', 'Data_Nascimento', 'Número', 'Email', 'Renda']
Feedback:  ['Id', 'Feedback']
Dados: ['1', 'Ana', '1995-06-15', '123456789', 'ana@email.com', 'R$5000']
Feedback:  ['1', 'Ótimo serviço!']
Dados: ['2', 'Bruno', '1989-03-22', '987654321', 'bruno@email.com', 'R$6000']
Feedback:  ['2', 'Poderia ser melhor.']
Dados: ['3', 'Melissa', '2001-12-08', '555555555', 'melissa@email.com', 'R$7000']
Feedback:  ['3', 'Excelente atendimento!']
Dados: ['4', 'Carlos', '1978-09-30', '111111111', 'carlos@email.com', 'R$8000']
Feedback:  ['4', 'Não estou satisfeito.']
Dados: ['5', 'Fernanda', '1992-07-14', '222222222', 'fernanda@email.com', 'R$9000']
Feedback:  ['5', 'Muito bom!']
Dados: ['6', 'Caio', '', '333333333', 'caio@email.com', 'R$10000']
Feedback:  ['6', 'Nenhum feedback.']

DataFrame de Feedbacks:
   Id                Feedback
0   1          Ótimo serviço!
1   2     Poderia ser melhor.
2   3  Excelente atendimento!
3   4   Nã

In [21]:
dt_atual     = dt_limpo.set_index("Id")          # ← Id vira index, não é dropado

dt_feedbacks = dt.df_feedbacks.set_index("Id")   # ← vem do atributo da instância

# Merge
dt_merged = dt_atual.join(dt_feedbacks, how="left")
print(dt_merged)

        Nome Data_Nascimento     Número               Email    Renda  \
Id                                                                     
1        Ana      1995-06-15  123456789       ana@email.com   5000.0   
2      Bruno      1989-03-22  987654321     bruno@email.com   6000.0   
3    Melissa      2001-12-08  555555555   melissa@email.com   7000.0   
4     Carlos      1978-09-30  111111111    carlos@email.com   8000.0   
5   Fernanda      1992-07-14  222222222  fernanda@email.com   9000.0   
6       Caio   Não Informado  333333333      caio@email.com  10000.0   

                  Feedback  
Id                          
1           Ótimo serviço!  
2      Poderia ser melhor.  
3   Excelente atendimento!  
4    Não estou satisfeito.  
5               Muito bom!  
6         Nenhum feedback.  
